# Thermal Concentric-C Acuity — A Guided Walkthrough

**Project:** `thermal-c-acuity` · *Rescue Rover Team — Concentric-C-pattern-Research*

---

## What this project is about

A thermal camera looking at a **concentric two-level "C" target** has to answer one
question: *can I still recognise the target, or has it dissolved into blur and noise?*

That single question is the whole project. Everything else is machinery for asking it
in a controlled way.

The target is two C-shaped rings cut from the same annulus — an outer C and a smaller
inner C, sharing a centre and sharing the direction their openings face. A complete
circle is easy: it is a closed blob. A **C** is not — you have to be able to see that
the ring is *broken*, and broken in the *same direction* on both levels. That is a much
harder discrimination, and it degrades gracefully as physics gets in the way.

## The pipeline, in the order we will build it

```
1. generate_target.py     →  draw the C pattern as an ideal bitmap
        ↓
2. thermal_model.py       →  turn geometry into temperature (°C)
        ↓
3. camera_model.py        →  put it through optics: IFOV, distance, blur
        ↓
4. noise_model.py         →  add NETD-style thermal noise
        ↓
5. preprocessing.py       →  temperature → binary mask
        ↓
6. c_detector.py          →  decide: is this a two-level concentric C?
        ↓
7. acuity_test.py         →  repeat the whole loop across distance, measure detection rate
        ↓
8. experiments/           →  isolate one degradation variable at a time
```

Steps 1–6 are the **simulation + detection core**. Steps 7–8 are the **experiments**
that use that core to produce numbers.

---

### How to read this notebook

Every section follows the same shape:

1. **Why this stage exists** — the physical or logical reason it has to be here.
2. **The code**, annotated.
3. **A live run** showing the actual image/numbers produced.

The notebook is **self-contained**: it does not import the project modules. Everything is
redefined here so you can run it top-to-bottom in one kernel and see every intermediate
state. The module files remain the source of truth for the experiments.

> **Note on one inconsistency in the repo:** the target geometry differs between files —
> `generate_target.py` and `thermal_model.py` use pixel geometry (`radius=150 / width=30`,
> a 500×500 canvas), while `c_detector.py` and all experiments use
> `radius=30 / width=6` on a 320×256 canvas. This notebook uses the **detector/experiment
> geometry**, because that is the one the actual acuity results are built on.

---
# 0 · Setup

Only four dependencies, and two of them are just for display.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib import patches
from pathlib import Path

# Make every figure a comfortable size and readable by default.
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.titlesize"] = 11

# DejaVu Sans (matplotlib's default) has no glyph for the ✅ / ❌ marks used in
# figure titles, which produces "Glyph missing from font" warnings. Segoe UI
# ships with Windows and does have them, so put it in front as a fallback.
for _f in ["C:/Windows/Fonts/segoeui.ttf", "C:/Windows/Fonts/arial.ttf"]:
    if Path(_f).exists():
        from matplotlib import font_manager
        font_manager.fontManager.addfont(_f)
        plt.rcParams["font.family"] = "sans-serif"
        plt.rcParams["font.sans-serif"] = [font_manager.FontProperties(fname=_f).get_name(),
                                           "DejaVu Sans"]
        break

# Every random draw in this notebook comes from this one generator, so the
# whole notebook is reproducible run-to-run.
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

print("numpy   ", np.__version__)
print("opencv  ", cv2.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("font     ", plt.rcParams["font.sans-serif"][0])

: 

## Shared constants

These values are the project's canonical numbers. They appear in
`c_detector.py`, in every experiment, and in the acuity test — so pinning them here in
one place is what makes the rest of the notebook easy to reason about.

| Constant | Value | Meaning |
|---|---|---|
| `WIDTH`, `HEIGHT` | 320, 256 | simulated sensor resolution (px) |
| `BACKGROUND_TEMP` | 25.0 °C | ambient scene temperature |
| `TARGET_TEMP` | 40.0 °C | hot target temperature |
| `CONTRAST` | 15.0 °C | thermal contrast ΔT |
| `OUTER_RADIUS`, `OUTER_WIDTH` | 30, 6 px | outer C (reference geometry) |
| `INNER_RADIUS`, `INNER_WIDTH` | 15, 4 px | inner C (exactly half the outer) |
| `GAP_ANGLE` | 45° | angular size of each C opening |
| `THRESHOLD` | 80 / 255 | binarisation cut (≈ 29.7 °C) |
| `KERNEL_SIZE` | 3 | morphological opening kernel |

In [ ]:
# ---- Sensor ---------------------------------------------------------------
WIDTH, HEIGHT = 320, 256

# ---- Scene temperatures ---------------------------------------------------
BACKGROUND_TEMP = 25.0     # °C — ambient
TARGET_TEMP     = 40.0     # °C — hot target
CONTRAST        = TARGET_TEMP - BACKGROUND_TEMP   # 15 °C

# ---- Target geometry (reference / near-distance) --------------------------
CX, CY = WIDTH // 2, HEIGHT // 2

OUTER_RADIUS, OUTER_WIDTH = 30.0, 6.0
INNER_RADIUS, INNER_WIDTH = 15.0, 4.0     # note: exactly half the outer
GAP_ANGLE = 45.0                          # degrees

# ---- Preprocessing --------------------------------------------------------
THRESHOLD   = 80      # in 8-bit intensity
KERNEL_SIZE = 3

print(f"ΔT = {CONTRAST:.1f} °C   "
      f"outer ring: r={OUTER_RADIUS:.0f}±{OUTER_WIDTH/2:.0f} px   "
      f"inner: r={INNER_RADIUS:.0f}±{INNER_WIDTH/2:.0f} px")

---
# 1 · `generate_target.py` — Drawing the C pattern

## Why this stage exists

Before anybody can claim a detector works, the target has to be **defined exactly**.
Not photographed — *defined*. A synthetic target is the only way to know the ground
truth: you know where the centre is, how thick the ring is, how wide the gap is, and
therefore you can say precisely what "detected" ought to mean.

This is the standard approach in EO/IR system modelling. You build a target with known
geometry, push it through a simulated imaging chain, and measure what survives.

## The geometry trick

The whole pattern is drawn with **two polar predicates** over a coordinate grid:

1. **Radius test** — is this pixel at the right distance from the centre to be inside an
   annulus of width `w` centred on radius `r`?
   ```
   ring = (r - w/2) ≤ distance ≤ (r + w/2)
   ```
2. **Angle test** — is this pixel inside the angular sector we want to *remove* to
   create the opening?
   ```
   gap = 0° ≤ angle ≤ gap_angle
   ```

A pixel belongs to the C if it passes **both** the ring test and the *negation* of the
gap test:

```
C = ring AND NOT gap
```

Vectorised over the whole image with `np.ogrid`, so each C is one boolean array
operation — no loops, no drawing primitives, no anti-aliasing. The edges are hard.

Two details that matter later:

- **The gap starts at 0°**, which is the `+x` axis pointing right. So the opening of the
  C always faces right in image coordinates. We will exploit that orientation when the
  detector checks that both levels open in the same direction.
- **`np.arctan2(y - cy, x - cx)`** in image coordinates (y down, x right) means angles
  increase *clockwise* on screen. Not a problem — just be consistent.

### The `draw_c` function

Identical logic appears in `generate_target.py`, `thermal_model.py`, `camera_model.py`
and `c_detector.py`. In `generate_target.py` it writes `255` into a `uint8` bitmap (an
idealised image); everywhere else it writes a temperature. Same mask, different payload.

In [ ]:
def draw_c_mask(cx, cy, radius, width, gap_angle=GAP_ANGLE,
                height=HEIGHT, width_img=WIDTH):
    """Return a boolean mask of a C-shaped ring.

    The mask is 1 where the pixel is part of the C, 0 elsewhere.

    Parameters
    ----------
    cx, cy     : centre of the ring, in pixels
    radius     : centre-line radius of the annulus, in pixels
    width      : radial thickness of the annulus, in pixels
    gap_angle  : angular size of the opening, in degrees (starts at 0° = +x axis)
    """
    # --- coordinate grid ---------------------------------------------------
    y, x = np.ogrid[:height, :width_img]

    # --- radial distance of every pixel from the centre --------------------
    distance = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)

    # --- predicate 1: is the pixel inside the annulus? ---------------------
    ring = (distance >= radius - width / 2) & (distance <= radius + width / 2)

    # --- angular position of every pixel, normalised to [0, 360) -----------
    angle = np.degrees(np.arctan2(y - cy, x - cx))
    angle = (angle + 360.0) % 360.0

    # --- predicate 2: is the pixel inside the opening? ---------------------
    gap = (angle >= 0.0) & (angle <= gap_angle)

    # --- C = ring minus gap ------------------------------------------------
    return ring & ~gap

### Building the two-level target

The inner C is **exactly half** the outer in both radius and width
(30/15 px, 6/4 px). The ratio `30/15 = 2.0` is far above the detector's
`RADIUS_RATIO_MIN = 1.3`, so the two levels are comfortably distinguishable even after
the geometry is blurred and shrunk.

Both Cs use the **same `gap_angle`**, which is what makes them read as one design
rather than two unrelated arcs.

In [ ]:
def make_two_level_c_mask():
    """Outer C + inner C, both opening toward 0° (+x axis)."""
    outer = draw_c_mask(CX, CY, OUTER_RADIUS, OUTER_WIDTH)
    inner = draw_c_mask(CX, CY, INNER_RADIUS, INNER_WIDTH)
    return outer | inner


mask = make_two_level_c_mask()

# `generate_target.py` stores this as a uint8 bitmap with 255 = "target".
target_bitmap = (mask * 255).astype(np.uint8)

covered = int(mask.sum())
print(f"target pixels : {covered}  ({100 * covered / mask.size:.2f}% of frame)")
print(f"bitmap dtype  : {target_bitmap.dtype}, unique values: {np.unique(target_bitmap)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

# --- 1. ideal bitmap (what generate_target.py produces) -------------------
axes[0].imshow(target_bitmap, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Ideal bitmap — 255 = target\n(generate_target.py output)")

# --- 2. the two masks separately -----------------------------------------
outer = draw_c_mask(CX, CY, OUTER_RADIUS, OUTER_WIDTH)
inner = draw_c_mask(CX, CY, INNER_RADIUS, INNER_WIDTH)

axes[1].imshow(outer, cmap="Blues", alpha=0.85)
axes[1].imshow(inner, cmap="Oranges", alpha=0.85)
axes[1].set_title("Outer C (blue) and inner C (orange)\ninner = exactly half the outer")

# --- 3. geometry annotation ----------------------------------------------
axes[2].imshow(target_bitmap, cmap="gray", vmin=0, vmax=255)
axes[2].add_patch(patches.Circle((CX, CY), OUTER_RADIUS,
                                 fill=False, ec="#d62728", lw=1.4, ls="--"))
axes[2].add_patch(patches.Circle((CX, CY), INNER_RADIUS,
                                 fill=False, ec="#2ca02c", lw=1.4, ls="--"))
axes[2].annotate("", xy=(CX + OUTER_RADIUS, CY), xytext=(CX, CY),
                 arrowprops=dict(arrowstyle="->", color="#d62728", lw=1.6))
axes[2].text(CX + 3, CY - 4, "r_out = 30 px", color="#d62728", fontsize=9)
axes[2].text(CX - 0.5, CY + 22, "r_in = 15 px", color="#2ca02c", fontsize=9)

# Mark the opening: the gap spans 0°..45°, i.e. from the +x axis rotating down
theta = np.radians(np.linspace(0, GAP_ANGLE, 40))
axes[2].plot(CX + 42 * np.cos(theta), CY + 42 * np.sin(theta),
             color="#ff7f0e", lw=2.5)
axes[2].text(CX + 30, CY + 40, "gap 45°", color="#ff7f0e", fontsize=10)

axes[2].set_title("Geometry: radii and the 45° opening")

for ax in axes:
    ax.set_xlim(CX - 55, CX + 55)
    ax.set_ylim(CY + 55, CY - 55)
    ax.axis("off")

plt.tight_layout()
plt.show()

**What to notice:** the opening faces right on *both* rings, and the rings are clean
binary edges. This is the ground truth. Everything that follows is degradation of this
one image.

---
# 2 · `thermal_model.py` — From geometry to temperature

## Why this stage exists

The bitmap was a *shape*. A thermal camera does not see shapes — it sees **radiance**,
which we approximate as **temperature**. So the next step is to give the pattern a
physical payload: 25 °C everywhere, 40 °C on the target.

This is where the number that governs the entire project enters: **thermal contrast
ΔT**. Detection difficulty is not driven by the pattern alone, it is driven by *how much
hotter* the pattern is than its surroundings.

## Why temperature, not `uint8`, all the way through

Notice that the modules keep the **float32 temperature map** through the whole
simulation chain — distance, blur, noise — and only convert to `uint8` at the very last
step, inside `preprocessing.py`.

That ordering is deliberate and correct:

| If you quantise early | Because we quantise late |
|---|---|
| 15 °C of contrast becomes 255 levels → 0.059 °C per level, coarse | the physics stays in real units |
| blur on an 8-bit image loses precision twice | Gaussian blur is linear, so it is exact in float |
| a 0.3 °C noise σ is invisible below 1 LSB | noise of any σ is representable |
| you cannot change exposure afterwards | you can rescale `min_temp`/`max_temp` freely |

Only when we need a *binary decision* do we collapse temperature to 8 bits and
threshold.

## The model

It is deliberately simple:

```
temperature(x, y) = 25.0 °C              everywhere
temperature(x, y) = 40.0 °C              where the C mask is 1
```

No emissivity term, no atmospheric transmission, no path radiance. For an acuity study
what matters is that ΔT is *known and controllable* — a full radiometric model would add
parameters without adding discriminating power.

In [ ]:
def make_temperature_map(target_temp=TARGET_TEMP,
                         background_temp=BACKGROUND_TEMP,
                         radius_scale=1.0):
    """Two-level concentric C rendered as a °C temperature map.

    `radius_scale` multiplies both radii and widths — used later for
    spatial-resolution sweeps.
    """
    temperature = np.full((HEIGHT, WIDTH), background_temp, dtype=np.float32)

    outer = draw_c_mask(CX, CY,
                        OUTER_RADIUS * radius_scale,
                        OUTER_WIDTH * radius_scale)
    inner = draw_c_mask(CX, CY,
                        INNER_RADIUS * radius_scale,
                        INNER_WIDTH * radius_scale)

    temperature[outer | inner] = target_temp
    return temperature


temperature = make_temperature_map()

print(f"temperature map : {temperature.shape}, dtype={temperature.dtype}")
print(f"range           : {temperature.min():.2f} .. {temperature.max():.2f} °C")
print(f"unique values   : {np.unique(temperature)}")
print(f"target area     : {int((temperature > BACKGROUND_TEMP).sum())} px")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

# --- 1. temperature map, physical colour scale ---------------------------
im = axes[0].imshow(temperature, cmap="inferno",
                    vmin=BACKGROUND_TEMP, vmax=TARGET_TEMP)
plt.colorbar(im, ax=axes[0], label="Temperature (°C)", fraction=0.046)
axes[0].set_title("Temperature map\n25 °C background, 40 °C target")

# --- 2. the same thing as a histogram ------------------------------------
axes[1].hist(temperature.ravel(), bins=40, color="#c44e52", edgecolor="white")
axes[1].set_yscale("log")
axes[1].set_xlabel("Temperature (°C)")
axes[1].set_ylabel("Pixel count (log)")
axes[1].set_title("Bimodal by construction: background vs target\n"
                  "(the gap between them *is* ΔT)")
axes[1].grid(alpha=0.3)

# --- 3. a horizontal cut through the centre ------------------------------
row = temperature[CY, :]
axes[2].plot(row, color="#4c72b0", lw=1.4)
axes[2].axhline(BACKGROUND_TEMP, ls=":", color="gray", lw=1)
axes[2].axhline(TARGET_TEMP, ls=":", color="gray", lw=1)
axes[2].axhspan(BACKGROUND_TEMP, TARGET_TEMP, color="#ffd9a0", alpha=0.35)
axes[2].annotate(f"ΔT = {CONTRAST:.0f} °C",
                 xy=(20, (BACKGROUND_TEMP + TARGET_TEMP) / 2),
                 fontsize=11, color="#b36b00")
axes[2].set_xlabel("x (px)")
axes[2].set_ylabel("Temperature (°C)")
axes[2].set_title(f"Horizontal profile through y = {CY}")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

The profile is a clean staircase: background, then a 30 px rise to the outer ring,
background again in the annulus between the two Cs, then the inner ring. **Every edge in
that staircase is information the detector will later try to recover** — and every one of
them is what blur and noise will attack.

---
# 3 · `camera_model.py` — Optics, IFOV and distance

## Why this stage exists

So far the target has been specified in **pixels**. But pixels are not a physical unit.
A pixel is an *angle* — and the true size of a pixel on the target depends on how far
away the target is.

This is the step that makes acuity meaningful. Two things happen as the camera backs
away from the target:

1. **The target shrinks in pixels** — it subtends the same angle from a smaller piece of
   the FOV, so fewer pixels cover it. This is pure geometric magnification.
2. **The blur grows in pixels** — optical/diffraction/aberration blur subtends a roughly
   fixed solid angle, so as the target shrinks under it, the blur eats a larger *fraction*
   of the target.

Both effects are the same underlying phenomenon — *the target is being sampled by fewer
and fewer resolution elements* — which is exactly what an acuity test ought to measure.

## The key camera parameter: IFOV

**IFOV** (Instantaneous Field Of View) is the angle one pixel subtends:

```
IFOV = pixel_pitch / focal_length        [radians]
```

With this project's numbers:

| Parameter | Value | Note |
|---|---|---|
| `PIXEL_PITCH` | 12 µm | 12e-6 m |
| `FOCAL_LENGTH` | 6 mm | 6e-3 m |
| `IFOV` | 2.0 mrad | = 12e-6 / 6e-3 |

**2 mrad per pixel.** Sanity check: the full 320-px horizontal FOV is
`320 × 2 mrad = 0.64 rad ≈ 36.7°`. That is a plausible wide-angle thermal lens. Good.

## From metres to pixels

A pixel that subtends 2 mrad covers a width on the target plane of:

```
mm_per_pixel(d) = d · 1000 · IFOV        [mm at distance d metres]
```

So a 60 mm object seen from 1 m covers `60 / (1·1000·0.002) = 30 px`. From 4 m it covers
**7.5 px**. From 8 m, under 4 px.

That shrinkage is the whole distance story. Note the module's physical target —
`OUTER_RADIUS_MM = 60`, `INNER_RADIUS_MM = 30` — is chosen so that **at 1 m the target
lands at exactly 30 px**, which is the reference geometry we used in sections 1–2.

In [ ]:
# ============================================================================
# Camera specification (from camera_model.py)
# ============================================================================
PIXEL_PITCH   = 12e-6        # m
FOCAL_LENGTH  = 6e-3         # m

IFOV = PIXEL_PITCH / FOCAL_LENGTH               # radians per pixel

# ============================================================================
# Physical target dimensions (from camera_model.py)
# ============================================================================
OUTER_RADIUS_MM = 60.0
OUTER_WIDTH_MM  = 12.0
INNER_RADIUS_MM = 30.0
INNER_WIDTH_MM  = 8.0

# ============================================================================
# Conversions
# ============================================================================
def mm_per_pixel(distance_m):
    """Ground-plane size of one pixel at a given range, in mm."""
    return distance_m * 1000.0 * IFOV


def mm_to_pixels(size_mm, distance_m):
    """Angular size of a physical object, expressed in pixels."""
    return size_mm / mm_per_pixel(distance_m)


print(f"IFOV            : {IFOV:.3e} rad = {IFOV * 1000:.3f} mrad/pixel")
print(f"Horizontal FOV  : {WIDTH * IFOV * 1000:.1f} mrad "
      f"= {np.degrees(WIDTH * IFOV):.1f}°")
print(f"Vertical FOV    : {np.degrees(HEIGHT * IFOV):.1f}°")
print()
print("Apparent size of the outer C (120 mm diameter) vs range:")
print()
print(f"{'distance':>10} {'mm/pixel':>10} {'outer r (px)':>14} {'outer w (px)':>14} {'ring (px)':>11}")
print("-" * 63)
for d in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 6.0, 8.0]:
    r_px = mm_to_pixels(OUTER_RADIUS_MM, d)
    w_px = mm_to_pixels(OUTER_WIDTH_MM, d)
    print(f"{d:>8.1f} m {mm_per_pixel(d):>9.2f} {r_px:>13.1f} {w_px:>13.1f} {r_px:>10.1f}")

### The physical / pixel duality

There are two equivalent ways to model "the camera moved further away", and the project
uses both — in different files:

| Approach | Used in | Mechanism |
|---|---|---|
| **Physical** | `camera_model.py` | place a mm-sized target at range `d`, convert to px via IFOV |
| **Relative** | `acuity_test.py`, `experiments/distance` | rescale the reference target by `scale = d_ref / d` |

The relative version is just the physical version pre-evaluated: if the reference
distance is 0.5 m and the geometry is *fixed in pixels* there, then moving to `d` metres
scales the pixel size by `0.5 / d`. Same physics, no trigonometry, faster to sweep.

### The blur model

`camera_model.py` applies a fixed `GaussianBlur(5, 5, sigmaX=0.0)` — sigma is derived
from the kernel, giving σ ≈ 1.1 px. But `acuity_test.py` models blur **growing with
distance**, which is the more realistic choice:

```
σ(d) = BASE_BLUR_SIGMA + BLUR_PER_METER · (d − REFERENCE_DISTANCE)
     = 0.5 + 0.8 · (d − 0.5)          [pixels]
```

| d | σ (px) |
|---|---|
| 0.5 m | 0.5 |
| 1.0 m | 0.9 |
| 2.0 m | 1.7 |
| 4.0 m | 3.3 |

Displayed in the notebook below: the *same* template resampled to progressively smaller
pixel footprints, then blurred, which is what the camera actually delivers.

In [ ]:
# ============================================================================
# Distance simulation by template rescaling (the acuity_test.py approach)
# ============================================================================
REFERENCE_DISTANCE = 0.5      # m — at this range the target has its reference size
BASE_BLUR_SIGMA    = 0.5      # px — optical blur at the reference distance
BLUR_PER_METER     = 0.8      # px per metre of additional range


def simulate_distance(temperature, distance, reference_distance=REFERENCE_DISTANCE):
    """Crop the target, rescale it by reference_distance/distance, recentre.

    This models ONLY the geometric shrink — no blur, no noise.
    """
    height, width = temperature.shape
    scale = reference_distance / distance

    # locate the target
    target_mask = temperature > BACKGROUND_TEMP
    ys, xs = np.where(target_mask)
    if len(xs) == 0:
        return temperature.copy()

    target = temperature[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

    new_w = max(2, int(target.shape[1] * scale))
    new_h = max(2, int(target.shape[0] * scale))

    # INTER_AREA is the correct choice for *downsampling*: it area-averages, so a
    # shrinking ring is integrated rather than point-sampled (which would alias).
    resized = cv2.resize(target, (new_w, new_h), interpolation=cv2.INTER_AREA)

    result = np.full((height, width), BACKGROUND_TEMP, dtype=np.float32)

    # recentre with clipping so the target never falls off-frame at short range
    start_x, start_y = width // 2 - new_w // 2, height // 2 - new_h // 2
    end_x, end_y = start_x + new_w, start_y + new_h

    sx1, sy1, sx2, sy2 = 0, 0, new_w, new_h
    dx1, dy1, dx2, dy2 = start_x, start_y, end_x, end_y

    if dx1 < 0:
        sx1, dx1 = -dx1, 0
    if dy1 < 0:
        sy1, dy1 = -dy1, 0
    if dx2 > width:
        sx2 -= (dx2 - width); dx2 = width
    if dy2 > height:
        sy2 -= (dy2 - height); dy2 = height

    result[dy1:dy2, dx1:dx2] = resized[sy1:sy2, sx1:sx2]
    return result


def simulate_blur(temperature, distance):
    """Gaussian blur whose σ grows linearly with range."""
    sigma = BASE_BLUR_SIGMA + BLUR_PER_METER * (distance - REFERENCE_DISTANCE)
    sigma = max(0.1, sigma)
    # kernel size (0,0) => OpenCV derives it from sigma. Correct for a
    # continuous-σ model; a fixed odd kernel would quantise σ badly at large values.
    return cv2.GaussianBlur(temperature, (0, 0), sigmaX=sigma, sigmaY=sigma)


# --- how big is the target at each range, in pixels? ----------------------
base = make_temperature_map()
print(f"{'range':>8} {'target px':>11} {'ring width':>12} {'blur σ':>9}")
print("-" * 44)
for d in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0]:
    scaled = simulate_distance(base, d)
    n = int((scaled > BACKGROUND_TEMP).sum())
    sigma = BASE_BLUR_SIGMA + BLUR_PER_METER * (d - REFERENCE_DISTANCE)
    # outer ring width in px at this range = 6 px * scale
    ring_px = OUTER_WIDTH * REFERENCE_DISTANCE / d
    print(f"{d:>6.1f} m {n:>10} px {ring_px:>11.1f} px {sigma:>8.2f} px")

In [ ]:
distances = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0]

fig, axes = plt.subplots(2, len(distances), figsize=(15, 5.0))

for i, d in enumerate(distances):
    ax = axes[0, i]
    scaled = simulate_distance(base, d)
    im = ax.imshow(scaled, cmap="inferno", vmin=BACKGROUND_TEMP, vmax=TARGET_TEMP)
    n = int((scaled > BACKGROUND_TEMP).sum())
    ax.set_title(f"d = {d:.1f} m\n{n} px", fontsize=9)
    ax.axis("off")

    ax = axes[1, i]
    blurred = simulate_blur(scaled, d)
    sigma = BASE_BLUR_SIGMA + BLUR_PER_METER * (d - REFERENCE_DISTANCE)
    ax.imshow(blurred, cmap="inferno", vmin=BACKGROUND_TEMP, vmax=TARGET_TEMP)
    ax.set_title(f"σ = {sigma:.2f} px", fontsize=9)
    ax.axis("off")

axes[0, 0].set_ylabel("scaled only", fontsize=11)
axes[1, 0].set_ylabel("+ distance blur", fontsize=11)

fig.suptitle("Two effects of range: the target shrinks in pixels, and the blur grows "
             "in pixels — so the blur consumes a larger fraction of the target",
             fontsize=11)
plt.tight_layout()
plt.show()

**Top row** is pure geometry — the template is resampled smaller. The rings still exist,
they are just made of fewer pixels.

**Bottom row** adds the range-dependent blur. Watch the outer ring: at 0.5 m it is 6 px
wide under a 0.5 px blur (comfortable, ~12:1). At 4 m it is 0.75 px wide under a 3.3 px
blur — the blur is now **four times wider than the feature it is supposed to resolve**.
That is the physical death of the pattern, and it happens well before the target
disappears from the frame.

This ratio — *feature width ÷ blur width* — is the real acuity metric. Distance is just
the convenient dial that changes it.

---
# 4 · `noise_model.py` — Thermal noise (NETD)

## Why this stage exists

Blur is a *deterministic* loss of information: given the same image, it always removes
the same detail. Noise is different — it is *stochastic*. It does not remove the signal,
it makes the signal **unreliable**: the same target sometimes crosses the threshold,
sometimes does not.

That distinction is why the acuity test in section 7 runs **20 trials per distance**. A
single run against noise tells you almost nothing; the quantity of interest is the
**detection rate** — the probability that the detector gets the right answer.

## The model

```python
noise = np.random.normal(loc=0.0, scale=noise_std, size=shape)   # °C
noisy = temperature + noise
```

Gaussian, zero-mean, i.i.d. per pixel, expressed **in °C**.

This is the standard **NETD** abstraction — Noise Equivalent Temperature Difference, the
σ of temperature fluctuation a thermal detector exhibits. Real NETD is a per-frame,
temporal + spatial mix; modelling it as additive Gaussian in temperature captures the
essential behaviour for a threshold-based detector.

## Why `scale` must be in °C, not LSBs

Because we deliberately quantise *late* (section 2), σ can be specified in physical
units and applies uniformly across the simulation. A σ of 0.3 °C against a 15 °C contrast
means the noise is **2% of the signal** — small. A σ of 5 °C is **33% of the signal** —
brutal. Those are the two ends of the noise experiment.

Here is the crucial consequence for the *detector*: the threshold sits at 80/255, i.e.
**29.7 °C**, which is 4.7 °C above background. A pixel that is really background can be
pushed over the threshold by a `+4.7 °C` fluctuation — about **16σ** at σ = 0.3 °C
(essentially never) but only **1.6σ** at σ = 3 °C (happens all the time, ~5% of pixels).
Noise does not blur the shape; it **sprinkles salt over it**, and morphological opening
in section 5 is the countermeasure.

In [ ]:
import math

NOISE_SIGMA = 0.30      # °C — the project default used by acuity_test.py


def add_thermal_noise(temperature_map, noise_std=NOISE_SIGMA, generator=None):
    """Add zero-mean Gaussian temperature noise (NETD model)."""
    if generator is None:
        generator = np.random.default_rng()
    noise = generator.normal(loc=0.0, scale=noise_std, size=temperature_map.shape)
    return (temperature_map + noise).astype(np.float32)


# ---------------------------------------------------------------------------
# How likely is noise to push a background pixel across the threshold?
# ---------------------------------------------------------------------------
threshold_temp = BACKGROUND_TEMP + (THRESHOLD / 255.0) * CONTRAST
margin = threshold_temp - BACKGROUND_TEMP

print(f"threshold sits at          : {threshold_temp:.2f} °C")
print(f"margin above background     : {margin:.2f} °C")
print()
print(f"{'noise σ':>9} {'margin/σ':>10} {'P(background flips)':>21}")
print("-" * 42)
for s in NOISE_SIGMA * np.array([0.5, 1, 3, 10, 16]):
    # one-sided Gaussian tail: P(background pixel noise exceeds the margin)
    p = 0.5 * (1 - math.erf((margin / s) / np.sqrt(2)))
    print(f"{s:>7.2f}°C {margin / s:>9.1f}σ {p:>20.2%}")

In [ ]:
noise_levels = [0.0, 1.0, 3.0, 5.0]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.9))

for ax, s in zip(axes, noise_levels):
    if s == 0.0:
        noisy = base.copy()
        label = "σ = 0 °C\n(no noise)"
    else:
        noisy = add_thermal_noise(base, s, np.random.default_rng(7))
        label = f"σ = {s:.1f} °C"

    im = ax.imshow(noisy, cmap="inferno", vmin=BACKGROUND_TEMP - 6,
                   vmax=TARGET_TEMP + 6)
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(label, fontsize=10)
    ax.axis("off")

fig.suptitle("Additive Gaussian temperature noise at fixed geometry "
             "(note: the colour scale is deliberately widened to show the noise)",
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# --- why a threshold detector cares: pixel-value distributions -------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: how much of the frame is background vs target
axes[0].hist(base.ravel(), bins=60, color="#4c72b0", edgecolor="white")
axes[0].axvline(threshold_temp, color="#d62728", lw=2, ls="--")
axes[0].text(threshold_temp + 0.4, axes[0].get_ylim()[1] * 0.6,
             f"threshold\n{threshold_temp:.1f} °C", color="#d62728", fontsize=9)
axes[0].set_yscale("log")
axes[0].set_xlabel("Temperature (°C)")
axes[0].set_ylabel("Pixel count (log)")
axes[0].set_title("Clean image: a clean gap around the threshold")
axes[0].grid(alpha=0.3)

# Right: same, after noise -- the background tail reaches the threshold
noisy = add_thermal_noise(base, 3.0, np.random.default_rng(3))
axes[1].hist(noisy.ravel(), bins=60, color="#c44e52", edgecolor="white", alpha=0.8)
axes[1].axvline(threshold_temp, color="#d62728", lw=2, ls="--")
axes[1].set_yscale("log")
axes[1].set_xlabel("Temperature (°C)")
axes[1].set_ylabel("Pixel count (log)")
axes[1].set_title("σ = 3 °C: background now spills past the threshold\n"
                  "→ false positives (the reason morphology exists)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

The right-hand histogram is the motivation for the entire next section. With σ = 3 °C
there is a visible population of **background pixels above the threshold**. Those become
spurious white specks, and `findContours` will happily turn them into tiny contours that
pollute the detector's candidate list.

---
# 5 · `preprocessing.py` — Temperature → clean binary mask

## Why this stage exists

The detector in section 6 does not reason about temperatures. It reasons about
**contours** — connected white regions in a binary image. So something has to bridge the
gap between the float32 temperature map and a binary mask, and that bridge has to make
hard decisions.

Preprocessing is that bridge, and it has exactly three steps.

```
temperature (°C)
      ↓  normalise + quantise
8-bit thermal image (0..255)
      ↓  threshold
binary image (0 or 255)
      ↓  morphological opening
cleaned binary image
```

## Step 1 — Temperature → 8-bit

A linear affine map, then clip:

```
image = clip((T − min_temp) / (max_temp − min_temp), 0, 1) × 255
```

With `min_temp = 25`, `max_temp = 40`, the mapping is
**6.8 (instrument units) per °C**. Anything below 25 °C saturates to 0, anything above
40 °C saturates to 255.

This saturation is not a bug — it is the simulation's implicit AGC. It means **the
threshold is *relative* to the stated window**, which is why the contrast experiment in
section 8(a) has to recompute the threshold per target temperature rather than reusing
80.

## Step 2 — Threshold

`cv2.threshold(image, 80, 255, THRESH_BINARY)` — everything > 80 becomes 255, everything
else 0.

**80/255 = 0.314**, so in temperature: `25 + 0.314 × 15 = **29.7 °C**`. The threshold sits
at almost exactly **one third of the way** from background to target — clearly above the
background, well below the target, and deliberately *asymmetric*: closer to background
than to target. That gives noise more room before it corrupts the target and slightly
more sensitivity to the target's lower edge as blur softens it.

## Step 3 — Morphological opening

```
opening = erode(k) then dilate(k)
```

Opening removes any structure **smaller than the kernel** (3×3, so features thinner than
~3 px) while leaving larger structures geometrically unchanged.

| Noise creates | Opening removes |
|---|---|
| isolated hot pixels | yes — 1–2 px specks vanish |
| thin bridges between regions | yes |
| ragged 1-px edges on real rings | yes |

It is the *right* operator here in a way that closing or a median filter would not be:
the failure mode is **false structure**, not missing structure, so we want to delete small
things rather than fill small holes.

**Caveat worth knowing:** opening is not free. It slightly erodes thin features. As the
target shrinks under blur (section 3), the ring gets thinner and eventually a
3×3 opening starts eating the ring itself. That interacts directly with the
spatial-resolution experiment.

## Step 4 (comment-only) — the disabled pipeline

Note that `preprocessing.py` (and its copy inside `experiments/resolution/`) has a large
block of ~240 lines **commented out** at the top — an earlier draft of the same functions
plus a visualisation script. The live code starts at the second `import cv2`. It is dead
weight rather than a bug, but it triples the apparent size of the file, which is why the
pipeline looks more intimidating than it is.

In [ ]:
def temperature_to_uint8(temperature, min_temp, max_temp):
    """Linear temperature → 8-bit mapping, clipped to [0, 255]."""
    normalised = (temperature - min_temp) / (max_temp - min_temp)
    normalised = np.clip(normalised, 0, 1)
    return (normalised * 255).astype(np.uint8)


def threshold_thermal_image(image, threshold):
    """Binary threshold. Returns a 0/255 uint8 mask."""
    _, binary = cv2.threshold(image, threshold, 255, cv2.THRESH_BINARY)
    return binary


def clean_binary_image(binary_image, kernel_size=KERNEL_SIZE):
    """Morphological opening: erode then dilate, removes features < kernel."""
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    return cv2.morphologyEx(binary_image, cv2.MORPH_OPEN, kernel)


def preprocess(temperature, min_temp=BACKGROUND_TEMP, max_temp=TARGET_TEMP,
               threshold=THRESHOLD, kernel_size=KERNEL_SIZE):
    """Full chain: °C map → cleaned binary mask."""
    thermal_image = temperature_to_uint8(temperature, min_temp, max_temp)
    binary_image  = threshold_thermal_image(thermal_image, threshold)
    cleaned_image = clean_binary_image(binary_image, kernel_size)
    return cleaned_image


# --- run it on the clean target -------------------------------------------
thermal = temperature_to_uint8(base, BACKGROUND_TEMP, TARGET_TEMP)
binary = threshold_thermal_image(thermal, THRESHOLD)
cleaned = preprocess(base)

thr_c = BACKGROUND_TEMP + (THRESHOLD / 255) * (TARGET_TEMP - BACKGROUND_TEMP)
print(f"temperature → uint8 : {thermal.dtype}, {thermal.min()}..{thermal.max()}")
print(f"threshold           : {THRESHOLD}/255  ⇒  {thr_c:.2f} °C")
print(f"binary white pixels : {int((binary > 0).sum())}")
print(f"cleaned white pixels: {int((cleaned > 0).sum())}")
print(f"pixels removed      : {int((binary > 0).sum()) - int((cleaned > 0).sum())}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4.1))

im = axes[0].imshow(thermal, cmap="inferno", vmin=0, vmax=255)
plt.colorbar(im, ax=axes[0], fraction=0.046, label="intensity")
axes[0].set_title(f"1. Temperature → uint8\n(0 = 25 °C, 255 = 40 °C)")

axes[1].imshow(binary, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"2. Threshold @ {THRESHOLD}/255\n({thr_c:.1f} °C)")

axes[2].imshow(cleaned, cmap="gray", vmin=0, vmax=255)
axes[2].set_title(f"3. Morphological opening ({KERNEL_SIZE}×{KERNEL_SIZE})\n"
                  "→ cleaned binary mask")

# Difference view: what the opening actually changed
diff = np.zeros((*binary.shape, 3), dtype=np.uint8)
diff[..., 0] = binary                      # red = removed by opening
diff[..., 1] = cleaned                     # green = kept
axes[3].imshow(diff)
axes[3].set_title("Overlay — red: removed by opening\ngreen/yellow: retained")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# --- now the interesting part: preprocessing under noise ------------------
noise_cases = [0.0, 1.0, 3.0, 5.0]

fig, axes = plt.subplots(2, 4, figsize=(15, 7))

for i, s in enumerate(noise_cases):
    if s == 0.0:
        temp = base.copy()
    else:
        temp = add_thermal_noise(base, s, np.random.default_rng(11))

    bin_raw = threshold_thermal_image(
        temperature_to_uint8(temp, BACKGROUND_TEMP, TARGET_TEMP), THRESHOLD)
    bin_clean = preprocess(temp)

    axes[0, i].imshow(bin_raw, cmap="gray", vmin=0, vmax=255)
    axes[0, i].set_title(f"σ = {s:.1f} °C — raw threshold\n"
                         f"{int((bin_raw > 0).sum())} px", fontsize=9)
    axes[0, i].axis("off")

    axes[1, i].imshow(bin_clean, cmap="gray", vmin=0, vmax=255)
    axes[1, i].set_title(f"after opening\n{int((bin_clean > 0).sum())} px", fontsize=9)
    axes[1, i].axis("off")

fig.suptitle("Morphological opening is what keeps the mask usable as noise rises — "
             "but note it also erodes the thin rings", fontsize=11)
plt.tight_layout()
plt.show()

**Progression:** at σ = 0 the opening changes essentially nothing (as it should). At
σ = 1–3 °C the raw threshold sprouts hundreds of specks and the opening wipes them out —
this is the opening earning its place. At σ = 5 °C the noise is dense enough that specks
begin to *merge* into blobs bigger than the kernel, and the morphology starts losing the
battle while simultaneously thinning the real rings.

That is the honest picture: morphology is a filter with a fixed scale, and it works
exactly as long as the noise blobs stay smaller than the signal features.

---
# 6 · `c_detector.py` — Is this *actually* a two-level concentric C?

## Why this stage exists

A mask with white rings in it is not a detection. The rings could be circles, arcs, a
smudge, or three unrelated blobs. The detector's job is to answer a **specific** yes/no
question:

> *Are there two nested, roughly concentric contours, the outer at least 1.3× the radius
> of the inner, and does each of them have an angular gap of at least 20°, with the two
> gaps pointing the same way?*

Every one of those clauses is a defence against a particular way of being wrong. This is
the intellectual core of the project — the simulation exists to test **this function**.

## The four gates

| # | Gate | Threshold | What it rejects |
|---|---|---|---|
| 1 | `is_c` | gap ≥ **20°** | complete circles, blobs, solid shapes |
| 2 | `same_center` | centres within **5 px** | two Cs that merely happen to be nearby |
| 3 | `different_levels` | radius ratio ≥ **1.3** | a single ring split into two contours by noise |
| 4 | `same_gap_direction` | gap centres within **15°** | two Cs that open in *different* directions |

A pair must pass **all four**. Any failure → `None` → "not detected".

## The interesting gate: measuring the angular gap

The naive way to find the opening is to look at the contour's own points. That is
fragile: contour extraction under noise produces points that wander, and a partially
occluded or lightly blurred ring may not have a clean contour at all.

`find_image_angular_gap` does something better — it **probes the image radially**:

1. Take 360 angles, one per degree.
2. For each angle, walk out from the centre along that ray from `radius − tol` to
   `radius + tol`.
3. If **any** pixel along that ray is white, mark the angle *occupied*.

This gives a 360-element boolean "occupancy ring". Then the gap is just the **longest run
of `False`**.

Two implementation details that are easy to get wrong and are handled correctly here:

- **Wrap-around at 0°/360°.** The occupancy array is doubled (`np.concatenate([occ, occ])`)
  before the run-length scan, so a gap straddling the 0° boundary is found as one run
  rather than two halves. The run is then clamped to 360 bins.
- **`ring_tolerance` scales with radius** — `max(2, int(radius * 0.25))`. A fixed
  tolerance would either miss the ring on a large contour or swallow the background on a
  small one. 25% of the radius is a sensible radial search window.

Note the gap here is **0°–45°**, i.e. it touches the wrap boundary. So the wrap handling
is not a theoretical nicety in this project — it is exercised on every single run.

In [ ]:
# ============================================================================
# Detector configuration (from c_detector.py)
# ============================================================================
CENTER_TOLERANCE        = 5.0     # px
RADIUS_RATIO_MIN        = 1.3
MIN_GAP_ANGLE           = 20.0    # degrees
GAP_DIRECTION_TOLERANCE = 15.0    # degrees


def find_image_angular_gap(binary_image, center, radius, ring_tolerance, num_angles=360):
    """Probe the image radially and return (gap_angle_deg, gap_center_deg).

    For each angle, walk out from the centre between radius-tol and radius+tol.
    An angle counts as OCCUPIED if any pixel on that ray is white.
    """
    height, width = binary_image.shape
    cx, cy = center

    occupied = np.zeros(num_angles, dtype=bool)

    for i in range(num_angles):
        angle = 360.0 * i / num_angles
        theta = np.radians(angle)

        r_start = max(1, int(radius - ring_tolerance))
        r_end   = int(radius + ring_tolerance)

        for r in range(r_start, r_end + 1):
            x = int(round(cx + r * np.cos(theta)))
            y = int(round(cy + r * np.sin(theta)))

            if x < 0 or x >= width or y < 0 or y >= height:
                continue

            if binary_image[y, x] > 0:
                occupied[i] = True
                break

    # --- find the longest run of unoccupied angles, handling 0/360 wrap -----
    circular = np.concatenate([occupied, occupied])   # <-- the wrap fix

    longest_empty = current_empty = longest_end = 0
    for i, is_occupied in enumerate(circular):
        if not is_occupied:
            current_empty += 1
            if current_empty > longest_empty:
                longest_empty = current_empty
                longest_end = i
        else:
            current_empty = 0

    longest_empty = min(longest_empty, num_angles)     # clamp to one revolution

    gap_angle = longest_empty * 360.0 / num_angles
    gap_center_index = (longest_end - longest_empty / 2) % num_angles
    gap_center_angle = gap_center_index * 360.0 / num_angles

    return gap_angle, gap_center_angle


def is_c_shape(binary_image, center, radius, ring_tolerance,
               min_gap_angle=MIN_GAP_ANGLE):
    """Gate 1 — is there a sufficiently large angular gap?"""
    gap_angle, gap_center = find_image_angular_gap(
        binary_image, center, radius, ring_tolerance)
    return {
        "is_c": gap_angle >= min_gap_angle,
        "gap_angle": gap_angle,
        "gap_center": gap_center,
    }

### Visualising the occupancy ring

Before trusting the gap measurement, let's *see* it. The polar plot below is the
occupancy array itself: angle around the circle, and the radial extent of the target at
that angle. A C shows up as an obvious missing wedge.

In [ ]:
def occupancy_polar(binary_image, center, radius, ring_tolerance, num_angles=720):
    """Return the angles that are occupied, for polar plotting."""
    height, width = binary_image.shape
    cx, cy = center
    occ = np.zeros(num_angles, dtype=bool)

    for i in range(num_angles):
        theta = np.radians(360.0 * i / num_angles)
        for r in range(max(1, int(radius - ring_tolerance)),
                       int(radius + ring_tolerance) + 1):
            x, y = int(round(cx + r * np.cos(theta))), int(round(cy + r * np.sin(theta)))
            if 0 <= x < width and 0 <= y < height and binary_image[y, x] > 0:
                occ[i] = True
                break
    return occ


fig = plt.figure(figsize=(14, 4.6))

# What the detector sees
ax = fig.add_subplot(1, 3, 1)
ax.imshow(cleaned, cmap="gray", vmin=0, vmax=255)
ax.add_patch(patches.Circle((CX, CY), OUTER_RADIUS, fill=False,
                            ec="#00d0ff", lw=1.5, ls=":"))
ax.add_patch(patches.Circle((CX, CY), INNER_RADIUS, fill=False,
                            ec="#00ff88", lw=1.5, ls=":"))
ax.set_title("Cleaned mask + detector search rings")
ax.set_xlim(CX - 55, CX + 55); ax.set_ylim(CY + 55, CY - 55); ax.axis("off")

# Occupancy ring, outer — polar
occ = occupancy_polar(cleaned, (CX, CY), OUTER_RADIUS, max(2, int(OUTER_RADIUS * 0.25)))
angles = np.radians(np.arange(len(occ)) * 360 / len(occ))

ax = fig.add_subplot(1, 3, 2, projection="polar")
ax.fill_between(angles, 0, occ.astype(float), color="#4c72b0", alpha=0.7)
ax.set_title("Outer ring occupancy (polar)\n"
             "the missing wedge IS the C opening", fontsize=9, pad=14)
ax.set_yticklabels([])
ax.set_theta_zero_location("E")      # 0° at the +x axis, matching the code
ax.set_theta_direction(-1)           # image coords: angle increases clockwise

ax = fig.add_subplot(1, 3, 3)
gap_angle, gap_center = find_image_angular_gap(
    cleaned, (CX, CY), OUTER_RADIUS, max(2, int(OUTER_RADIUS * 0.25)))
ax.plot(occ.astype(int), lw=1.6, color="#4c72b0")
ax.fill_between(range(len(occ)), 0, occ.astype(int), color="#4c72b0", alpha=0.35)
ax.set_title(f"Same data, unwrapped\n"
             f"measured gap = {gap_angle:.0f}°, centre at {gap_center:.0f}°", fontsize=9)
ax.set_xlabel("angle (deg)")
ax.set_yticks([0, 1]); ax.set_yticklabels(["empty", "occupied"])
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Outer C : gap = {gap_angle:5.1f}°  centre = {gap_center:5.1f}°   "
      f"→ is_c = {gap_angle >= MIN_GAP_ANGLE}")
print(f"(ground truth: gap 45°, centre 22.5°)")

The measurement recovers the ground truth almost exactly. Now the full detector.

### `detect_concentric_pair` — the four gates

The structure is: build a candidate for every contour, then test **every pair** against
the four gates, and return the first pair that passes all four.

In [ ]:
def detect_concentric_pair(binary_image, contours,
                           center_tolerance=CENTER_TOLERANCE,
                           radius_ratio_min=RADIUS_RATIO_MIN,
                           min_gap_angle=MIN_GAP_ANGLE,
                           gap_direction_tolerance=GAP_DIRECTION_TOLERANCE):
    """Return a dict describing the two-level C, or None."""
    candidates = []

    # ---- build candidates from every contour ------------------------------
    for contour in contours:
        area = cv2.contourArea(contour)
        if area <= 0:
            continue

        (x, y), radius = cv2.minEnclosingCircle(contour)

        M = cv2.moments(contour)
        if M["m00"] == 0:
            continue                       # degenerate: no area, no centroid
        center = np.array([M["m10"] / M["m00"], M["m01"] / M["m00"]])

        # radial search window scales with the contour size
        ring_tolerance = max(2, int(radius * 0.25))

        c_info = is_c_shape(binary_image, center, radius, ring_tolerance, min_gap_angle)

        candidates.append({
            "contour": contour, "center": center, "radius": radius, "area": area,
            "is_c": c_info["is_c"],
            "gap_angle": c_info["gap_angle"],
            "gap_center": c_info["gap_center"],
        })

    if len(candidates) < 2:
        return None                        # need at least two rings

    # ---- test every pair --------------------------------------------------
    for i in range(len(candidates)):
        for j in range(i + 1, len(candidates)):
            c1, c2 = candidates[i], candidates[j]

            # GATE 1: both must be C-shaped
            if not c1["is_c"] or not c2["is_c"]:
                continue

            # GATE 2: shared centre
            center_distance = np.linalg.norm(c1["center"] - c2["center"])
            same_center = center_distance <= center_tolerance

            # GATE 3: distinct radii
            larger, smaller = max(c1["radius"], c2["radius"]), min(c1["radius"], c2["radius"])
            if smaller == 0:
                continue
            radius_ratio = larger / smaller
            different_levels = radius_ratio >= radius_ratio_min

            # GATE 4: openings pointing the same way
            gap_difference = abs(c1["gap_center"] - c2["gap_center"])
            gap_difference = min(gap_difference, 360 - gap_difference)   # circular
            same_gap_direction = gap_difference <= gap_direction_tolerance

            if same_center and different_levels and same_gap_direction:
                outer, inner = (c1, c2) if c1["radius"] > c2["radius"] else (c2, c1)
                return {
                    "outer": outer, "inner": inner,
                    "center_distance": center_distance,
                    "radius_ratio": radius_ratio,
                    "outer_gap_angle": outer["gap_angle"],
                    "inner_gap_angle": inner["gap_angle"],
                    "outer_gap_direction": outer["gap_center"],
                    "inner_gap_direction": inner["gap_center"],
                    "gap_direction_difference": gap_difference,
                }

    return None


def run_detector(mask):
    """Mask → (detected, contours, result)."""
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    result = detect_concentric_pair(mask, contours)
    return result is not None, contours, result

### Test 1 — the positive case

Does it fire on the real two-level C?

In [ ]:
detected, contours, result = run_detector(cleaned)

print("=" * 62)
print("POSITIVE TEST — two-level concentric C")
print("=" * 62)
print(f"contours found : {len(contours)}")

if result is None:
    print("❌ not detected")
else:
    print(f"✅  DETECTED")
    print()
    print(f"  outer radius      : {result['outer']['radius']:6.2f} px   "
          f"(truth 33.0 = r + w/2)")
    print(f"  inner radius      : {result['inner']['radius']:6.2f} px   "
          f"(truth 17.0)")
    print(f"  center distance   : {result['center_distance']:6.2f} px   "
          f"(limit {CENTER_TOLERANCE})")
    print(f"  radius ratio      : {result['radius_ratio']:6.2f}      "
          f"(limit {RADIUS_RATIO_MIN})")
    print(f"  outer gap         : {result['outer_gap_angle']:6.1f}°      "
          f"(truth 45°)")
    print(f"  inner gap         : {result['inner_gap_angle']:6.1f}°      "
          f"(truth 45°)")
    print(f"  outer gap dir     : {result['outer_gap_direction']:6.1f}°      "
          f"(truth 22.5°)")
    print(f"  inner gap dir     : {result['inner_gap_direction']:6.1f}°      "
          f"(truth 22.5°)")
    print(f"  dir difference    : {result['gap_direction_difference']:6.1f}°      "
          f"(limit {GAP_DIRECTION_TOLERANCE})")

In [ ]:
# --- visualise the accepted detection -------------------------------------
vis = cv2.cvtColor(cleaned, cv2.COLOR_GRAY2BGR)

if result is not None:
    # highlight the two accepted contours
    cv2.drawContours(vis, [result["outer"]["contour"]], -1, (0, 0, 255), 2)
    cv2.drawContours(vis, [result["inner"]["contour"]], -1, (0, 255, 0), 2)
    # mark their centres and the opening direction
    for c, col in [(result["outer"], (0, 0, 255)), (result["inner"], (0, 255, 0))]:
        ctr = tuple(np.round(c["center"]).astype(int))
        cv2.circle(vis, ctr, 3, col, -1)
        th = np.radians(c["gap_center"])
        cv2.line(vis, ctr,
                 (int(ctr[0] + 50 * np.cos(th)), int(ctr[1] + 50 * np.sin(th))),
                 col, 1, cv2.LINE_AA)

vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))

axes[0].imshow(cleaned, cmap="gray")
axes[0].set_title("Cleaned binary mask")
axes[1].imshow(vis_rgb)
axes[1].set_title("Accepted pair: outer (red), inner (green)\n"
                  "rays show measured opening directions")
axes[2].bar(["center\ndist (px)", "radius\nratio", "dir diff\n(deg)"],
            [result["center_distance"], result["radius_ratio"],
             result["gap_direction_difference"]],
            color=["#4c72b0", "#dd8452", "#55a868"])
axes[2].axhline(CENTER_TOLERANCE, ls="--", color="#c44e52", lw=1.2)
axes[2].text(2.4, CENTER_TOLERANCE, f"  limit {CENTER_TOLERANCE:.0f}", color="#c44e52")
axes[2].set_title("Measured margins\n(each must clear its limit)")
axes[2].grid(alpha=0.3, axis="y")

for ax in axes[:2]:
    ax.set_xlim(CX - 55, CX + 55); ax.set_ylim(CY + 55, CY - 55); ax.axis("off")

plt.tight_layout()
plt.show()

### Test 2 — the negative case (the important one)

A detector that says "yes" is worthless unless it can also say "no". `c_detector.py`
ships a deliberate negative control: `create_two_level_circles()` builds two **complete**
concentric circles with no openings at all. Geometrically they share a centre and have
the same radii — so **gates 2 and 3 pass**. Only gate 1 (`is_c`) can reject them.

This is the single most valuable test in the codebase, because it isolates the one thing
that makes the target a C rather than a blob.

In [ ]:
def create_two_level_circles():
    """Negative control: two COMPLETE concentric circles, no opening."""
    temperature = np.full((HEIGHT, WIDTH), BACKGROUND_TEMP, dtype=np.float32)
    y, x = np.ogrid[:HEIGHT, :WIDTH]
    distance = np.sqrt((x - CX) ** 2 + (y - CY) ** 2)

    outer_circle = ((distance >= OUTER_RADIUS - OUTER_WIDTH / 2) &
                    (distance <= OUTER_RADIUS + OUTER_WIDTH / 2))
    inner_circle = ((distance >= INNER_RADIUS - INNER_WIDTH / 2) &
                    (distance <= INNER_RADIUS + INNER_WIDTH / 2))

    temperature[outer_circle] = TARGET_TEMP
    temperature[inner_circle] = TARGET_TEMP
    return temperature


circles_mask = preprocess(create_two_level_circles())
circles_detected, circles_contours, circles_result = run_detector(circles_mask)

# also measure the gap the detector finds on a full circle
gap_full, dir_full = find_image_angular_gap(
    circles_mask, (CX, CY), OUTER_RADIUS, max(2, int(OUTER_RADIUS * 0.25)))

print("=" * 62)
print("NEGATIVE TEST — two complete concentric CIRCLES")
print("=" * 62)
print(f"contours found          : {len(circles_contours)}")
print(f"measured outer gap      : {gap_full:.1f}°  (a full circle: ~0°)")
print(f"MIN_GAP_ANGLE           : {MIN_GAP_ANGLE:.1f}°")
print()
print(f"result                  : {'DETECTED (BAD!)' if circles_detected else 'rejected (correct)'}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].imshow(create_two_level_circles(), cmap="inferno",
               vmin=BACKGROUND_TEMP, vmax=TARGET_TEMP)
axes[0].set_title("Negative control\ntwo complete circles (temperature)")

axes[1].imshow(circles_mask, cmap="gray")
axes[1].set_title("After preprocessing")

occ_full = occupancy_polar(circles_mask, (CX, CY), OUTER_RADIUS,
                           max(2, int(OUTER_RADIUS * 0.25)))
ang_full = np.radians(np.arange(len(occ_full)) * 360 / len(occ_full))
ax = fig.add_subplot(1, 4, 3, projection="polar")
ax.fill_between(ang_full, 0, occ_full.astype(float), color="#c44e52", alpha=0.7)
ax.set_theta_zero_location("E"); ax.set_theta_direction(-1)
ax.set_yticklabels([])
ax.set_title("Occupancy: a FULL ring\nno wedge -> gate 1 fails", fontsize=9, pad=14)

# side-by-side comparison of measured gap
axes[3].bar(["two-level C", "complete circles"], [gap_angle, gap_full],
            color=["#55a868", "#c44e52"])
axes[3].axhline(MIN_GAP_ANGLE, ls="--", color="#333", lw=1.4)
axes[3].text(-0.45, MIN_GAP_ANGLE + 2, f"MIN_GAP_ANGLE = {MIN_GAP_ANGLE:.0f}°",
             fontsize=9)
axes[3].set_ylabel("measured gap (deg)")
axes[3].set_title("The discriminating measurement")
axes[3].grid(alpha=0.3, axis="y")

axes[0].axis("off"); axes[1].axis("off")

plt.tight_layout()
plt.show()

**This is the payoff of the whole section.** The two targets have identical centres and
identical radii — the *only* difference is the opening, and the gap measurement is the
only thing that separates them. Gate 1 does its job. (In the printed output above, the
`✅` / `❌` marks are console-only; the figures use plain text so the rendering is clean.)

### Where the detector's assumptions could break

Worth being explicit about the fragility, because the experiments probe exactly these:

1. **`ring_tolerance = 25% of radius`** — as the target shrinks under distance, the
   *radius* shrinks and so does the tolerance, but the blur σ does **not** shrink. At
   some range the tolerance window no longer spans the blurred ring. This is a coupling
   between the geometry and the blur model that will limit the maximum range.
2. **Hard `MIN_GAP_ANGLE = 20°`** — blur rounds the sharp 45° opening into a smooth
   gradient. The measured gap shrinks gradually rather than failing abruptly. A 20° floor
   against a nominal 45° gives 25° of headroom, which is generous but finite.
3. **Morphology erodes thin rings** — at high blur the rings get thin, and the 3×3
   opening starts removing ring pixels, which raises the measured gap artificially
   (a thin broken ring looks like it has *more* opening). Note this failure mode pushes
   toward **false positives**, not false negatives — the opposite of the intuitive
   direction.

---
# 7 · `acuity_test.py` — The end-to-end acuity measurement

## Why this stage exists

Sections 1–6 built a *simulator* and a *detector*. Neither produces a number. `acuity_test.py`
does — and it is the only file in the project that answers the actual research question:

> **How far away can the rover be and still reliably recognise this target?**

## The design decisions that matter

**1 · The full chain runs per trial.**

```
original target
   ↓  simulate_distance()      geometric shrink
   ↓  simulate_blur()          σ grows with range
   ↓  add_thermal_noise()      random draw
   ↓  preprocess()             → binary mask
   ↓  findContours()
   ↓  detect_concentric_pair() → yes / no
```

**2 · Detection rate, not a single answer.** `TRIALS_PER_DISTANCE = 20`. Every trial
draws fresh noise, so the result is a **probability**, not a verdict. This is the correct
way to test anything against a stochastic process — one run tells you nothing about a
0.7-probability event.

**3 · One RNG, seeded once, shared across all trials.**

```python
rng = np.random.default_rng(42)     # created ONCE before the distance loop
```

This is subtle but important. Because it is created once and passed down, the 20 trials
at 4 m use *different* noise from the 20 trials at 3 m — but the whole experiment is
reproducible run-to-run. Seeding *inside* the trial loop would give 20 different noises
but a different set each run.

**4 · An explicit reliability criterion.**

```python
RELIABLE_DETECTION_RATE = 80.0      # percent
```

A distance counts as "reliable" if ≥ 80% of trials detect. The 80% figure is a
convention (it appears in every experiment file), and it is worth being honest that it is
a *choice*, not a derived optimum. What it buys is a well-defined, comparable answer:
"reliable range = 2.5 m" means something specific and is comparable across experiments.

**5 · The sweep.**

| Parameter | Value |
|---|---|
| `DISTANCES` | 0.5 → 4.0 m, step 0.5 |
| `REFERENCE_DISTANCE` | 0.5 m (full size) |
| `TRIALS_PER_DISTANCE` | 20 |
| `BASE_BLUR_SIGMA` | 0.5 px |
| `BLUR_PER_METER` | 0.8 px/m |
| `NOISE_STD` | 0.30 °C |
| `RANDOM_SEED` | 42 |
| `RELIABLE_DETECTION_RATE` | 80 % |

Note the noise here is **0.30 °C — much milder than the 3–5 °C of section 4.** That is
not an accident: this test is measuring the *distance* limit, so noise is held near the
realistic NETD floor and distance is the variable. The noise experiment in section 8
does the opposite.

In [ ]:
# ============================================================================
# Acuity test configuration (from acuity_test.py)
# ============================================================================
DISTANCES              = np.arange(0.5, 4.1, 0.5)
TRIALS_PER_DISTANCE    = 20
RELIABLE_DETECTION_RATE = 80.0


def simulate_camera(original_temperature, distance, generator):
    """The full imaging chain for one trial."""
    scaled  = simulate_distance(original_temperature, distance)
    blurred = simulate_blur(scaled, distance)
    noisy   = add_thermal_noise(blurred, NOISE_SIGMA, generator)
    return noisy


def detect_target(temperature):
    """temperature → (detected, cleaned_mask, contours, result)."""
    mask = preprocess(temperature, min_temp=BACKGROUND_TEMP, max_temp=TARGET_TEMP,
                      threshold=THRESHOLD, kernel_size=KERNEL_SIZE)
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    result = detect_concentric_pair(mask, contours)
    return result is not None, mask, contours, result


def run_acuity_test():
    """Sweep distance, N trials each, return per-distance detection rates."""
    original = make_temperature_map()

    # ONE generator for the whole experiment → reproducible, but every trial
    # still gets a fresh noise realisation.
    generator = np.random.default_rng(RNG_SEED)

    results = []
    print("=" * 66)
    print("THERMAL C ACUITY — REPEATED TRIAL TEST")
    print("=" * 66)
    print(f"{'distance':>10} {'trials':>8} {'detected':>10} {'rate':>9}")
    print("-" * 66)

    for distance in DISTANCES:
        detected_count = 0
        for _ in range(TRIALS_PER_DISTANCE):
            simulated = simulate_camera(original, distance, generator)
            detected, mask, contours, result = detect_target(simulated)
            if detected:
                detected_count += 1

        rate = detected_count / TRIALS_PER_DISTANCE * 100.0
        results.append({"distance": float(distance),
                        "detected_count": detected_count,
                        "trials": TRIALS_PER_DISTANCE,
                        "detection_rate": rate})
        print(f"{distance:>8.1f} m {TRIALS_PER_DISTANCE:>8} "
              f"{detected_count:>10} {rate:>8.1f}%")

    return results


acuity_results = run_acuity_test()

In [ ]:
def find_reliable_distance(results, threshold=RELIABLE_DETECTION_RATE):
    """Maximum distance still meeting the reliability criterion."""
    ok = [r["distance"] for r in results if r["detection_rate"] >= threshold]
    return max(ok) if ok else None


reliable_distance = find_reliable_distance(acuity_results)

print("=" * 66)
print("FINAL RESULT")
print("=" * 66)
if reliable_distance is not None:
    print(f"Reliable detection distance ({RELIABLE_DETECTION_RATE:.0f}%): "
          f"{reliable_distance:.1f} m")
else:
    print(f"No tested distance reached {RELIABLE_DETECTION_RATE:.0f}%.")

In [ ]:
dists = [r["distance"] for r in acuity_results]
rates = [r["detection_rate"] for r in acuity_results]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8),
                         gridspec_kw={"width_ratios": [1.35, 1]})

# --- detection rate curve -------------------------------------------------
axes[0].plot(dists, rates, marker="o", lw=2, color="#4c72b0", markersize=7,
             markerfacecolor="white", markeredgewidth=2)
axes[0].axhline(RELIABLE_DETECTION_RATE, ls="--", color="#c44e52", lw=1.6,
                label=f"{RELIABLE_DETECTION_RATE:.0f}% reliable threshold")
if reliable_distance is not None:
    axes[0].axvspan(0, reliable_distance, color="#55a868", alpha=0.13,
                    label=f"reliable ≤ {reliable_distance:.1f} m")
axes[0].set_xlabel("Distance (m)")
axes[0].set_ylabel("Detection Rate (%)")
axes[0].set_ylim(-5, 108)
axes[0].set_title("Detection rate vs range — the acuity curve")
axes[0].grid(alpha=0.3)
axes[0].legend(loc="lower left", fontsize=9)

# --- why: companion plot of feature-size ÷ blur ---------------------------
for d in dists:
    pass
ring_px = [OUTER_WIDTH * REFERENCE_DISTANCE / d for d in dists]
sigma_px = [BASE_BLUR_SIGMA + BLUR_PER_METER * (d - REFERENCE_DISTANCE) for d in dists]
ratio = [r / s for r, s in zip(ring_px, sigma_px)]

axes[1].plot(dists, ratio, marker="s", color="#dd8452", lw=2)
axes[1].axhline(1.0, ls="--", color="#555", lw=1.4)
axes[1].text(0.6, 1.08, "ring width = blur width", fontsize=9, color="#555")
axes[1].set_yscale("log")
axes[1].set_xlabel("Distance (m)")
axes[1].set_ylabel("outer ring width ÷ blur σ  (log)")
axes[1].set_title("The physics behind the curve\n"
                  "fails once the blur outgrows the feature")
axes[1].grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

### Reading the curve

The detection rate starts at 100% close in and falls as range increases. Two things to
take from it:

**The shape is a step, not a slope.** Detection does not degrade gradually from 100% to
0% — it holds at 100%, half-works for exactly one interval, then drops to zero and stays
there. That is characteristic of a **threshold** discriminator: as long as the measured
gap stays above 20° the detector is nearly always right, and once it drops below, it is
nearly always wrong. The 20-trial averaging is what reveals the narrow transition band
rather than a single misleading pass/fail.

**Where this particular curve lands is worth noting.** The reliable range comes out at
**0.5 m** — the closest distance tested — with 1.0 m already at a coin-flip and
everything beyond it dead. That is *shorter* than the blur physics alone would predict,
and the mismatch is informative rather than a mistake. Running the controlled
experiments below in the same kernel gives experiment 2 (blur only, 0.5 m) surviving a
21×21 kernel, and experiment 1 (distance only) failing between 0.7 m and 0.8 m. So:

| Configuration | Survives to |
|---|---|
| blur only | very heavy blur (σ ≈ 3.5 px) |
| distance only | ~0.7 m |
| **both together** | **0.5 m** |

The combined chain is far more fragile than either component alone. That is a real
property of the system — **the failure modes compound multiplicatively** — but it also
means the acuity test's headline number should be read as the *joint* limit, not as an
attribution of which effect dominates. Picking the attribution apart is precisely what
section 8 exists for.

In [ ]:
# --- what each distance actually looks like to the detector --------------
example_distances = [0.5, 1.0, 2.0, 3.0, 4.0]
generator = np.random.default_rng(RNG_SEED)
original = make_temperature_map()

fig, axes = plt.subplots(3, len(example_distances), figsize=(15, 8.4))

for i, d in enumerate(example_distances):
    simulated = simulate_camera(original, d, generator)
    detected, mask, contours, result = detect_target(simulated)

    # row 1: what the camera delivers
    axes[0, i].imshow(simulated, cmap="inferno",
                      vmin=BACKGROUND_TEMP - 2, vmax=TARGET_TEMP + 2)
    axes[0, i].set_title(f"d = {d:.1f} m\nthermal image", fontsize=9)

    # row 2: what preprocessing leaves
    axes[1, i].imshow(mask, cmap="gray", vmin=0, vmax=255)
    axes[1, i].set_title(f"binary mask\n{int((mask>0).sum())} px", fontsize=9)

    # row 3: verdict
    vis = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    if result is not None:
        cv2.drawContours(vis, [result["outer"]["contour"]], -1, (0, 0, 255), 2)
        cv2.drawContours(vis, [result["inner"]["contour"]], -1, (0, 255, 0), 2)
        verdict = "DETECTED"
        col = "#1a7f37"
    else:
        verdict = "not detected"
        col = "#c44e52"
    axes[2, i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    if result is not None:
        axes[2, i].set_title(f"{verdict}\ngap = {result['outer_gap_angle']:.0f}°",
                             fontsize=9, color=col)
    else:
        axes[2, i].set_title(verdict, fontsize=9, color=col)

for row_labels, ax_row in zip(["thermal", "mask", "verdict"], axes):
    ax_row[0].set_ylabel(row_labels, fontsize=10)

for ax in axes.ravel():
    ax.set_xlim(CX - 55, CX + 55)
    ax.set_ylim(CY + 55, CY - 55)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

*(Trial 1 only — one individual outcome, not the average.)*

This is the most informative figure in the notebook: you can watch the target cross the
detector's thresholds in stages. The thermal image stays visually plausible well past the
point where the mask has become ambiguous, and the mask stays *shaped like a C* for a
while after the detector has stopped accepting it. **A human eye is more forgiving than
the detector, and the gap between the two is the honest finding of the project.**

---
# 8 · `experiments/` — Isolating one variable at a time

## Why this stage exists

The acuity test changes distance, which changes **three things at once**: target size,
blur, *and* (through the resampling) the effective geometry. If detection fails at 3 m,
you cannot say *which* of those caused it.

The `experiments/` folder fixes that by running a **controlled sweep of one variable at a
time**, with everything else pinned. This is basic experimental design and it is the
reason this folder exists separately from `acuity_test.py`.

## The design matrix

| Experiment | File | Variable | Fixed | Purpose |
|---|---|---|---|---|
| **1 · Distance** | `distance/distance_only_test.py` | distance | blur OFF, noise OFF | pure geometric shrink |
| **2 · Blur** | `blur_only_test.py` | blur kernel 1→21 | d = 0.5 m, noise OFF | pure PSF degradation |
| **3 · Noise** | `noise/noise_only_test.py` | σ 0→10 °C | d = 0.5 m, blur OFF | pure stochastic corruption |
| **4 · Contrast** | `contrast/contrast_only_test.py` | target T 40→26 °C | d = 0.5 m, blur OFF, noise OFF | pure ΔT effect |
| **5 · Spatial resolution** | `resolution/spatial_resolution.py` | outer radius 30→20 px | blur OFF, noise OFF | pure sampling limit |

## Two structural things worth flagging

**`experiments/blur_only_test.py` sits at the top level, not in a `blur/` folder.** The
other four each have their own directory; blur is a loose file. Given the user-facing
structure lists `blur` alongside the others, this is a packaging inconsistency rather
than a functional one — but it means the folder listing does not match the experiment
list.

**`experiments/resolution/` contains its own copy of `preprocessing.py`.** So there are
now *two* copies of the preprocessing module in the repo. They are currently identical,
which means it works — but a future edit to the main `preprocessing.py` will silently not
apply to the spatial-resolution experiment. This is the kind of thing that costs an
afternoon six months later.

**Experiment 5 also defines its own local `detect_concentric_pair`**, with a
**reduced** gate set — it checks only `same_center` and `different_levels`. It does
**not** check `is_c`, and it does not check gap direction. So experiment 5's
"detection" means *"two nested contours with a shared centre"*, not *"a two-level C"*.
That is a real semantic difference from the other experiments and from `c_detector.py`.

Now let's run each of them.

---

### 8 · Experiment 1 — Distance only

Blur and noise both off. The only thing changing is how many pixels the target occupies.
This isolates the pure **sampling** limit, and gives the ceiling that the other
experiments degrade from.

In [ ]:
def simulate_distance_only(temperature, distance, reference_distance=0.5):
    """Geometric shrink only — no blur, no noise. Same as simulate_distance."""
    return simulate_distance(temperature, distance, reference_distance)


DR_DISTANCES = np.arange(0.5, 1.51, 0.1)     # finer step near the transition
original = make_temperature_map()

dist_only = []
print("=" * 62)
print("EXPERIMENT 1 — DISTANCE ONLY   (blur OFF, noise OFF)")
print("=" * 62)

for d in DR_DISTANCES:
    scaled = simulate_distance_only(original, d)
    detected, mask, contours, result = detect_target(scaled)
    n_px = int((scaled > BACKGROUND_TEMP).sum())
    dist_only.append({"distance": float(d), "detected": detected, "pixels": n_px})

print(f"{'distance':>10} {'target px':>11} {'verdict':>14}")
print("-" * 62)
for r in dist_only:
    print(f"{r['distance']:>8.1f} m {r['pixels']:>11} "
          f"{'DETECTED' if r['detected'] else 'NOT DETECTED':>14}")

last_ok = [r["distance"] for r in dist_only if r["detected"]]
print()
print(f"Last detected distance: "
      f"{max(last_ok):.1f} m" if last_ok else "none detected")

### 8 · Experiment 2 — Blur only

Distance pinned at 0.5 m, noise off, and the Gaussian **kernel size** swept from 1 (no
blur) to 21. This is the cleanest possible measurement of how much PSF the C can absorb
before the detector loses it.

Note the kernel is used as `GaussianBlur(img, (k, k), sigmaX=0)` — passing `sigmaX=0`
tells OpenCV to *derive* σ from the kernel size, so σ ≈ 0.3·(k−1)/2 + 0.8·... in
practice roughly σ ≈ k/6 for large k. The exact relationship matters less than the
monotonic ordering: larger kernel = more blur.

In [ ]:
BLUR_KERNELS = [1, 3, 5, 7, 9, 11, 15, 21]


def apply_blur_kernel(temperature, kernel_size):
    """kernel_size = 1 means no blur."""
    if kernel_size == 1:
        return temperature.copy()
    return cv2.GaussianBlur(temperature, (kernel_size, kernel_size), sigmaX=0)


original = make_temperature_map()
blur_results = []

print("=" * 62)
print("EXPERIMENT 2 — BLUR ONLY   (d = 0.5 m, noise OFF)")
print("=" * 62)
print(f"{'kernel':>8} {'σ (px)':>9} {'verdict':>16}")
print("-" * 62)

for k in BLUR_KERNELS:
    blurred = apply_blur_kernel(original, k)
    detected, mask, contours, result = detect_target(blurred)
    sigma = 0.0 if k == 1 else 0.3 * ((k - 1) * 0.5 - 1) + 0.8   # OpenCV's rule
    blur_results.append({"kernel": k, "sigma": sigma, "detected": detected})
    name = "NO BLUR" if k == 1 else f"{k}x{k}"
    print(f"{name:>8} {sigma:>9.2f} "
          f"{'DETECTED' if detected else 'NOT DETECTED':>16}")

last_blur_ok = [r["kernel"] for r in blur_results if r["detected"]]
print()
if last_blur_ok:
    print(f"Largest blur still detected: {max(last_blur_ok)}x{max(last_blur_ok)}")

### 8 · Experiment 3 — Noise only

Distance 0.5 m, blur off, σ swept from **0 to 10 °C** in 20 trials per level. This is
where the detection **rate** becomes essential — a single trial at σ = 3 °C is a coin
flip, and reporting it as pass/fail would be meaningless.

Note this experiment **clips** the noisy map back into `[25, 40] °C`
(`np.clip(noisy, BACKGROUND_TEMP, TARGET_TEMP)`). That is a deliberate physical
constraint — the scene cannot go below ambient or above the target — but it has a
side effect worth knowing: clipping is a **non-linear** operation that *removes* the
extreme noise excursions. It therefore makes the detector's job slightly easier than an
unclipped model would, and the effect grows with σ. At σ = 10 °C, clipping is doing a lot
of the denoising before the experiment even starts.

In [ ]:
NOISE_LEVELS = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 7.0, 10.0]
TRIALS_PER_NOISE = 20


def add_thermal_noise_clipped(temperature, noise_std, generator):
    """Noise, then clip to the physically realisable [background, target] range."""
    noisy = add_thermal_noise(temperature, noise_std, generator)
    return np.clip(noisy, BACKGROUND_TEMP, TARGET_TEMP).astype(np.float32)


original = make_temperature_map()
generator = np.random.default_rng(RNG_SEED)
noise_results = []

print("=" * 62)
print("EXPERIMENT 3 — NOISE ONLY   (d = 0.5 m, blur OFF)")
print("=" * 62)
print(f"{'noise σ':>10} {'trials':>8} {'detected':>10} {'rate':>9}")
print("-" * 62)

for s in NOISE_LEVELS:
    count = 0
    for _ in range(TRIALS_PER_NOISE):
        noisy = add_thermal_noise_clipped(original, s, generator)
        detected, mask, contours, result = detect_target(noisy)
        if detected:
            count += 1
    rate = count / TRIALS_PER_NOISE * 100
    noise_results.append({"noise": s, "detected": count, "rate": rate})
    print(f"{s:>8.1f} °C {TRIALS_PER_NOISE:>8} {count:>10} {rate:>8.1f}%")

ok_noise = [r["noise"] for r in noise_results if r["rate"] >= 80]
print()
print(f"80% reliable noise limit: σ ≤ {max(ok_noise):.1f} °C"
      if ok_noise else "No tested noise level reached 80%.")

### 8 · Experiment 4 — Contrast only

Distance 0.5 m, blur off, noise off, background pinned at 25 °C and the **target
temperature** walked down from 40 °C to 26 °C. ΔT goes from 15 °C down to 1 °C.

This experiment contains the single most instructive implementation detail in the whole
repo. Look at `temperature_to_uint8` inside `contrast_only_test.py`:

```python
min_temp, max_temp = BACKGROUND_TEMP, 40.0     # window FIXED at 25..40

# ... and then the threshold is RECOMPUTED per target temperature:
midpoint_temp = (BACKGROUND_TEMP + target_temp) / 2
threshold = ((midpoint_temp - min_temp) / (max_temp - min_temp)) * 255
```

The normalisation window stays at **25–40 °C** while the binarisation threshold is
recomputed as the **midpoint between background and this trial's target**.

That is the correct choice, and the reason is visible in the docstring: a fixed threshold
of 80 corresponds to 29.7 °C, so a target at 28 °C would produce pixels at
`(28−25)/15 × 255 = 51` — *below the threshold* — and detection would fail **for a
reason that has nothing to do with whether the detector can see the contrast**. Every
target below ~29.7 °C would fail identically. The experiment would be measuring the
threshold's position, not the contrast limit.

Adapting the threshold to the midpoint isolates the variable properly: each trial gets
the *fairest possible* cut for its own contrast, and the measured limit is genuinely the
contrast limit.

**The caveat:** adapting the threshold assumes you always know the target's temperature
in advance. In a real deployment you do not. So this experiment measures an **optimistic
bound** on contrast performance — a best case with perfect prior knowledge. That is worth
stating explicitly whenever the result is quoted.

In [ ]:
TARGET_TEMPERATURES = [40.0, 38.0, 36.0, 34.0, 32.0, 30.0, 29.0, 28.0, 27.0, 26.0]
TRIALS_PER_CONTRAST = 20


def make_contrast_target(target_temp):
    """Two-level C at a specified target temperature; same geometry as before."""
    temperature = np.full((HEIGHT, WIDTH), BACKGROUND_TEMP, dtype=np.float32)
    y, x = np.ogrid[:HEIGHT, :WIDTH]
    distance = np.sqrt((x - CX) ** 2 + (y - CY) ** 2)

    angle = np.degrees(np.arctan2(y - CY, x - CX))
    angle = (angle + 360) % 360
    gap = (angle >= 0) & (angle <= 40)          # this experiment uses 40° when drawing

    outer = ((distance >= OUTER_RADIUS - OUTER_WIDTH / 2) &
             (distance <= OUTER_RADIUS + OUTER_WIDTH / 2) & ~gap)
    inner = ((distance >= INNER_RADIUS - INNER_WIDTH / 2) &
             (distance <= INNER_RADIUS + INNER_WIDTH / 2) & ~gap)

    temperature[outer | inner] = target_temp
    return temperature


def adaptive_threshold_uint8(target_temp, min_temp=BACKGROUND_TEMP, max_temp=40.0):
    """Fixed 25..40 normalisation window, threshold at the background/target midpoint."""
    midpoint = (BACKGROUND_TEMP + target_temp) / 2.0
    threshold = int(((midpoint - min_temp) / (max_temp - min_temp)) * 255)
    return threshold


contrast_results = []

print("=" * 70)
print("EXPERIMENT 4 — CONTRAST ONLY   (d = 0.5 m, blur OFF, noise OFF)")
print("=" * 70)
print(f"{'target':>8} {'ΔT':>7} {'threshold':>11} {'trials':>8} {'detected':>10} {'rate':>8}")
print("-" * 70)

for target_temp in TARGET_TEMPERATURES:
    thr = adaptive_threshold_uint8(target_temp)
    temperature = make_contrast_target(target_temp)

    count = 0
    for _ in range(TRIALS_PER_CONTRAST):
        mask = preprocess(temperature, min_temp=BACKGROUND_TEMP, max_temp=40.0,
                          threshold=thr, kernel_size=3)
        contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        if detect_concentric_pair(mask, contours) is not None:
            count += 1

    rate = count / TRIALS_PER_CONTRAST * 100
    dt = target_temp - BACKGROUND_TEMP
    contrast_results.append({"target_temp": target_temp, "contrast": dt,
                             "threshold": thr, "rate": rate})
    print(f"{target_temp:>6.1f} °C {dt:>5.1f} °C {thr:>10} {TRIALS_PER_CONTRAST:>8} "
          f"{count:>10} {rate:>7.1f}%")

ok_contrast = [r["contrast"] for r in contrast_results if r["rate"] >= 80]
print()
print(f"Minimum ΔT for 80% reliability: {min(ok_contrast):.1f} °C"
      if ok_contrast else "No tested contrast achieved 80%.")

### 8 · Experiment 5 — Spatial resolution

Geometry pinned, blur off, noise off, contrast 15 °C — and the **outer radius** shrunk
from 30 px down to 20 px in 1-px steps. Inner radius follows at a fixed ratio (0.5), as
do the ring widths (0.20 and 0.13 of the outer radius).

This isolates the **sampling** limit: how few pixels can a ring be made of before the
detector loses it?

**The important caveat, restated.** As noted earlier, this experiment calls its *own*
`detect_concentric_pair`, which checks only centre coincidence and radius ratio. It does
**not** verify that the contours are C-shaped. So a "detection" here means *"two nested
contours sharing a centre"*.

That makes the numbers **not directly comparable** to experiments 1–4 or to
`acuity_test.py`. It is measuring a weaker, easier criterion. Whether that is a bug or an
intentional simplification is not something the code says — but it is exactly the kind of
discrepancy that silently inflates a reported result if the two are put in the same table.

In [ ]:
OUTER_RADII = list(range(30, 19, -1))     # 30 down to 20
INNER_RATIO = 0.5
OUTER_WIDTH_RATIO = 0.20
INNER_WIDTH_RATIO = 0.13
TRIALS_RES = 20


def detect_nested_only(mask, center_tolerance=5.0, radius_ratio_min=1.3):
    """The experiment-5 detector: centre + ratio ONLY (no C-shape check)."""
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    candidates = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area <= 0:
            continue
        (x, y), radius = cv2.minEnclosingCircle(contour)
        M = cv2.moments(contour)
        if M["m00"] == 0:
            continue
        candidates.append({
            "center": np.array([M["m10"] / M["m00"], M["m01"] / M["m00"]]),
            "radius": radius,
        })

    if len(candidates) < 2:
        return None

    for i in range(len(candidates)):
        for j in range(i + 1, len(candidates)):
            c1, c2 = candidates[i], candidates[j]
            center_distance = np.linalg.norm(c1["center"] - c2["center"])
            larger, smaller = max(c1["radius"], c2["radius"]), min(c1["radius"], c2["radius"])
            if smaller == 0:
                continue
            ratio = larger / smaller
            if center_distance <= center_tolerance and ratio >= radius_ratio_min:
                return {"center_distance": center_distance, "radius_ratio": ratio}
    return None


resolution_results = []

print("=" * 68)
print("EXPERIMENT 5 — SPATIAL RESOLUTION   (blur OFF, noise OFF, ΔT = 15 °C)")
print("=" * 68)
print(f"{'outer r':>9} {'ring w':>8} {'trials':>8} {'detected':>10} {'rate':>8}")
print("-" * 68)

for radius in OUTER_RADII:
    outer_w = max(2, radius * OUTER_WIDTH_RATIO)
    inner_r = radius * INNER_RATIO
    inner_w = max(2, radius * INNER_WIDTH_RATIO)

    count = 0
    for _ in range(TRIALS_RES):
        temperature = np.full((HEIGHT, WIDTH), BACKGROUND_TEMP, dtype=np.float32)
        outer = draw_c_mask(CX, CY, radius, outer_w)
        inner = draw_c_mask(CX, CY, inner_r, inner_w)
        temperature[outer | inner] = TARGET_TEMP

        mask = preprocess(temperature, min_temp=25.0, max_temp=40.0,
                          threshold=80, kernel_size=3)
        if detect_nested_only(mask) is not None:
            count += 1

    rate = count / TRIALS_RES * 100
    resolution_results.append({"radius": radius, "rate": rate, "inner_r": inner_r})
    print(f"{radius:>7} px {outer_w:>6.1f} px {TRIALS_RES:>8} {count:>10} {rate:>7.1f}%")

ok_res = [r["radius"] for r in resolution_results if r["rate"] >= 80]
print()
print(f"80% reliable minimum outer radius: {min(ok_res)} px"
      if ok_res else "No tested size reached 80%.")

In [ ]:
# --- all five controlled experiments side by side -------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 8.2))
axes = axes.ravel()

# 1 · distance
axes[0].plot([r["distance"] for r in dist_only],
             [100 if r["detected"] else 0 for r in dist_only],
             marker="o", color="#4c72b0", lw=2)
axes[0].set_xlabel("Distance (m)"); axes[0].set_ylabel("Detection (%)")
axes[0].set_title("1 · Distance only\n(target shrinks, no blur/noise)", fontsize=10)
axes[0].set_ylim(-8, 108)

# 2 · blur
axes[1].plot([r["kernel"] for r in blur_results],
             [100 if r["detected"] else 0 for r in blur_results],
             marker="s", color="#dd8452", lw=2)
axes[1].set_xlabel("Gaussian kernel"); axes[1].set_ylabel("Detection (%)")
axes[1].set_title("2 · Blur only\n(PSF grows, everything else fixed)", fontsize=10)
axes[1].set_ylim(-8, 108)

# 3 · noise
axes[2].plot([r["noise"] for r in noise_results],
             [r["rate"] for r in noise_results],
             marker="^", color="#c44e52", lw=2)
axes[2].axhline(80, ls="--", color="#333", lw=1.2)
axes[2].set_xlabel("Noise σ (°C)"); axes[2].set_ylabel("Detection (%)")
axes[2].set_title("3 · Noise only\n(20 trials/level — a rate, not a verdict)", fontsize=10)
axes[2].set_ylim(-8, 108)

# 4 · contrast
axes[3].plot([r["contrast"] for r in contrast_results],
             [r["rate"] for r in contrast_results],
             marker="D", color="#8172b3", lw=2)
axes[3].axhline(80, ls="--", color="#333", lw=1.2)
axes[3].set_xlabel("ΔT (°C)"); axes[3].set_ylabel("Detection (%)")
axes[3].set_title("4 · Contrast only\n(adapted threshold — optimistic bound)",
                  fontsize=10)
axes[3].set_ylim(-8, 108)

# 5 · spatial resolution
axes[4].plot([r["radius"] for r in resolution_results],
             [r["rate"] for r in resolution_results],
             marker="v", color="#55a868", lw=2)
axes[4].axhline(80, ls="--", color="#333", lw=1.2)
axes[4].set_xlabel("Outer radius (px)"); axes[4].set_ylabel("Detection (%)")
axes[4].set_title("5 · Spatial resolution\n(weaker detector — see caveat)",
                  fontsize=10)
axes[4].set_ylim(-8, 108)

# 6 · summary: where each limit falls
axes[5].axis("off")
axes[5].set_title("Summary — the limit each experiment finds", fontsize=10)
lines = [
    ("1 · distance",   f"last detected at {max(last_ok):.1f} m" if last_ok else "—"),
    ("2 · blur",       f"up to {max(last_blur_ok)}x{max(last_blur_ok)} kernel" if last_blur_ok else "—"),
    ("3 · noise",      f"σ ≤ {max(ok_noise):.1f} °C" if ok_noise else "—"),
    ("4 · contrast",   f"ΔT ≥ {min(ok_contrast):.1f} °C" if ok_contrast else "—"),
    ("5 · resolution", f"r ≥ {min(ok_res)} px" if ok_res else "—"),
]
for i, (name, value) in enumerate(lines):
    y = 0.82 - i * 0.16
    axes[5].text(0.02, y, name, fontsize=11, fontweight="bold")
    axes[5].text(0.52, y, value, fontsize=11, color="#1a7f37")
axes[5].text(0.02, 0.02,
             "each number is the 80%-reliability limit\n"
             "with all other variables pinned",
             fontsize=9, style="italic", color="#666")

for ax in axes[:5]:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### What the five experiments together tell you

Each curve is a different axis of the same physical question — *how much information does
the detector need before the target stops being a target?*

| Axis | The physical quantity | What actually kills the target | 80% limit (as run) |
|---|---|---|---|
| distance | pixels on target | sampling — combined with the geometry | ~0.7 m |
| blur | PSF width vs feature width | blur exceeding the ring | survives 21×21 (σ ≈ 3.5 px) |
| noise | σ vs threshold margin | threshold crossings become frequent | σ ≤ 4.0 °C |
| contrast | ΔT vs threshold | signal falling to the quantisation floor | ΔT ≥ 1.0 °C |
| resolution | px per ring | sampling alone | r ≥ 24 px *(weaker detector — see caveat)* |

### The finding that surprised me, and it is the real result

Read across those rows and the story is **not** the intuitive one:

- **Blur alone barely matters.** With distance pinned, the detector survives a 21×21
  kernel — σ ≈ 3.5 px against a 6 px ring, blur more than half the feature width. The
  morphology plus the 20° gap floor are remarkably tolerant of a soft edge.
- **Noise alone barely matters.** σ ≤ 4.0 °C against a 15 °C contrast is a *very* large
  noise budget — remember from section 4 that at σ = 3 °C the raw threshold already
  produces 5.8% false-positive pixels.
- **Contrast alone barely matters.** Detection holds down to ΔT = 1.0 °C, which works
  only because the experiment recomputes the threshold adaptively (see the caveat above);
  with a fixed threshold it would collapse near 4.7 °C.
- **Distance alone is what fails first** — between 0.7 m and 0.8 m.

So the binding constraint is **range**, and specifically the *interaction*: distance
shrinks the target in pixels (6 px ring → 3 px at 1 m → 1.5 px at 2 m) *while*
simultaneously applying a blur that does not shrink at all. Each effect is individually
survivable; together they push the ring below the scale where morphology and the 20° gap
floor still work. **The failure is multiplicative, not additive** — which is exactly why
`acuity_test.py` dies at 0.5–1.0 m even though neither its own blur nor its own 0.3 °C
noise would be fatal in isolation.

Those numbers are worth pausing on before they are put in a report:

| Result | Why it is softer than it looks |
|---|---|
| distance ≤ 0.7 m | zero-noise idealisation — experiment 1 has **no noise at all**, so it is an *upper* bound on range |
| contrast ΔT ≥ 1.0 °C | adaptive threshold assumes the target temperature is known in advance — an *optimistic* bound |
| resolution r ≥ 24 px | uses the weaker centre+ratio-only detector — **not comparable** to the others |
| blur 21×21 survives | a single deterministic run, no trials, no noise |
| noise σ ≤ 4.0 °C | clipped noisy maps are partly denoised before detection |

None of these are wrong individually. But a table that mixed them without those footnotes
would be combining five different notions of what "detected" means — and that, more than
any single number, is the thing to guard against when the results get written up.

---
# 9 · The pipeline end to end

Putting all eight stages back together, with the transformation of the data at each step:

```
┌─────────────────────────────────────────────────────────────────────────┐
│  1  generate_target.py                                                  │
│     C mask  →  uint8 bitmap (0 / 255)                                   │
│     polar predicates:  ring = (r-w/2 ≤ |p-c| ≤ r+w/2)                   │
│                        gap  = (0° ≤ θ ≤ 45°)                            │
│                        C    = ring AND NOT gap                          │
└────────────────────────────────┬────────────────────────────────────────┘
                                 │  same mask, new payload
┌────────────────────────────────▼────────────────────────────────────────┐
│  2  thermal_model.py                                                    │
│     bitmap  →  float32 temperature map   [25 °C bg, 40 °C target]       │
│     ⚠ stays in °C through the whole sim; quantised only at the end     │
└────────────────────────────────┬────────────────────────────────────────┘
                                 │
┌────────────────────────────────▼────────────────────────────────────────┐
│  3  camera_model.py                                                     │
│     IFOV = 12 µm / 6 mm = 2.0 mrad/pixel                                │
│     distance  →  geometric shrink (INTER_AREA, scale = d_ref/d)         │
│     distance  →  σ = 0.5 + 0.8·(d − 0.5)  px  Gaussian blur             │
└────────────────────────────────┬────────────────────────────────────────┘
                                 │
┌────────────────────────────────▼────────────────────────────────────────┐
│  4  noise_model.py                                                      │
│     temperature += N(0, σ²)      per-pixel, i.i.d., in °C (NETD model)  │
└────────────────────────────────┬────────────────────────────────────────┘
                                 │
┌────────────────────────────────▼────────────────────────────────────────┐
│  5  preprocessing.py                                                    │
│     °C  →  uint8   (linear map over [25, 40] °C)                        │
│         →  binary  (threshold @ 80/255 = 29.7 °C)                       │
│         →  cleaned (morphological opening, 3×3)                         │
└────────────────────────────────┬────────────────────────────────────────┘
                                 │
┌────────────────────────────────▼────────────────────────────────────────┐
│  6  c_detector.py                                                       │
│     findContours(RETR_TREE)                                             │
│     per contour:  minEnclosingCircle + moments → radius, centre         │
│                   radial probe at 360 angles → angular gap              │
│     per PAIR:     is_c · same_center · different_levels · same_gap_dir  │
│     → dict  or  None                                                    │
└────────────────────────────────┬────────────────────────────────────────┘
                                 │
┌────────────────────────────────▼────────────────────────────────────────┐
│  7  acuity_test.py                                                      │
│     20 trials × 8 distances → detection rate → reliable range           │
│     → "reliable detection distance (80%)"                               │
└────────────────────────────────┬────────────────────────────────────────┘
                                 │
┌────────────────────────────────▼────────────────────────────────────────┐
│  8  experiments/                                                        │
│     distance · blur · noise · contrast · spatial resolution             │
│     one variable at a time → attribution of the failure mode            │
└─────────────────────────────────────────────────────────────────────────┘
```

## The design, in one paragraph

The project is a **closed-loop simulation of an acuity test**. The target is defined
analytically so ground truth is exact; the imaging chain degrades it with physically
motivated operations (angular sampling, PSF, NETD); preprocessing converts the physical
temperature field into a decision-ready binary mask; and the detector makes a
conservative four-gate decision. Because every stage is parameterised, the *same* code
produces the distance sweep, the blur sweep, the noise sweep, the contrast sweep and the
resolution sweep — and the controlled sweeps are what turn a single "it stopped working"
observation into a statement about **which** physical effect is responsible.

## The three findings worth carrying forward

**1 · The failure is multiplicative, not additive — and range is what triggers it.**
No single degradation is fatal on its own: the detector survives very heavy blur
(experiment 2, 21×21), very strong noise (experiment 3, σ ≤ 4 °C), and very low contrast
(experiment 4, ΔT ≥ 1 °C). But distance (experiment 1) fails between 0.7 m and 0.8 m —
because distance *shrinks the target in pixels and grows the blur in pixels at the same
time*. Each effect is survivable; the product is not. That is why `acuity_test.py`, which
runs the full chain, dies at 0.5–1.0 m.

**2 · The discriminator is fragile by design, and the negative test proves it works.**
The C-versus-circle decision rests entirely on gate 1's 20° gap floor. `create_two_level_circles()`
is geometrically identical except for the opening, and the detector rejects it correctly —
which is the evidence that the yes-answers mean something.

**3 · Not every result in the repo is directly comparable.**
Experiment 4's adaptive threshold is an *optimistic* bound (it assumes the target
temperature is known in advance). Experiment 5 uses a *weaker* detector (no C-shape
check) and its own copy of `preprocessing.py`. Experiment 2 reports a single noiseless
run rather than a rate. Each is defensible individually, but a table mixing all five
numbers would silently combine different definitions of "detected".

## Appendix · Issues found while writing this notebook

These are things the code does that are worth knowing, ordered by how much they could
bite. None of them break the notebook (which is self-contained), but they are live in the
repo.

| # | Where | Issue | Consequence |
|---|---|---|---|
| 1 | `generate_target.py`, `thermal_model.py` vs rest | geometry differs: `r=150/w=30` on 500×500 vs `r=30/w=6` on 320×256 | the first two files are disconnected from the detector's geometry; they are demo scripts, not pipeline stages |
| 2 | `experiments/resolution/preprocessing.py` | duplicate copy of `preprocessing.py` | a future fix to the main module will silently not apply here |
| 3 | `experiments/resolution/spatial_resolution.py` | local `detect_concentric_pair` omits the `is_c` and gap-direction gates | its detection rate uses a **weaker** criterion than all other experiments |
| 4 | `experiments/blur_only_test.py` | sits at the top level, not in a `blur/` folder | folder structure does not match the experiment list |
| 5 | `noise/noise_only_test.py` | clips noisy map to `[25, 40] °C` | clipping *denoises* the extremes, making high-σ results optimistically biased |
| 6 | `preprocessing.py` (both copies) | ~240 lines of commented-out earlier draft | file is ~3× longer than its live content; easy to misread |
| 7 | `thermal_model.py` | rebuilds `temperature` at the end and never exits the script cleanly | module has no functions — everything runs on import |
| 8 | `acuity_test.py` | imports `WIDTH`, `HEIGHT`, `BACKGROUND_TEMP` … from `c_detector.py` | module coupling: configuration lives in the detector, which then imports preprocessing |
| 9 | `c_detector.py` | module-level config executed on import; top-level demo runs only under `__main__` | importing it for constants is safe, but the constants are not in a dedicated config module |

**The one I would fix first:** #3. A detector that skips the C-shape check in exactly one
experiment means that experiment's headline number cannot go in the same column as the
others. Either import the real detector from `c_detector.py`, or rename the local
function to something like `detect_nested_pair` so the weaker contract is obvious at the
call site. The second option is what this notebook did.

---

*Notebook generated from the source in `thermal-c-acuity/`. Every cell is
self-contained — no imports from the project modules — so it runs top-to-bottom in a
single kernel. All randomness is seeded with `RNG_SEED = 42`, so the numbers you see here
are reproducible.*